# Generate Passive Greenhouse Indoor Datasets

This notebook creates synthetic indoor greenhouse datasets from outdoor weather data.
- Input: Daily outdoor weather data for 2024 and 2025
- Output: Hourly indoor greenhouse data (24 records/day) with passive greenhouse effects
- No active control systems - only natural greenhouse behavior

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Define Processing Functions

In [2]:
def load_and_standardize(filepath):
    """
    Load CSV and standardize column names.
    """
    df = pd.read_csv(filepath)
    print(f"  Loaded {len(df)} daily records from {Path(filepath).name}")
    
    # Parse datetime
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # Standardize column names
    df = df.rename(columns={
        'temp': 'outdoor_temp',
        'humidity': 'outdoor_humidity',
        'windspeed': 'outdoor_windspeed',
        'solarradiation': 'solar'
    })
    
    # Keep only required columns
    required_cols = ['datetime', 'datetimeEpoch', 'outdoor_temp', 'outdoor_humidity', 
                     'outdoor_windspeed', 'solar']
    
    # Add sunrise/sunset if available
    if 'sunriseEpoch' in df.columns:
        required_cols.extend(['sunriseEpoch', 'sunsetEpoch'])
    
    df = df[required_cols].copy()
    
    return df

In [3]:
def expand_to_hourly(df):
    """
    Expand daily data to hourly resolution (24 records per day).
    Interpolate values across hours.
    """
    # Create complete hourly range
    min_date = df['datetime'].min().normalize()
    max_date = df['datetime'].max().normalize() + timedelta(days=1)
    hourly_range = pd.date_range(start=min_date, end=max_date, freq='H')[:-1]  # Exclude last point
    
    print(f"  Creating hourly index: {hourly_range[0]} to {hourly_range[-1]}")
    print(f"  Total hourly records: {len(hourly_range)}")
    
    # Create hourly dataframe
    hourly_df = pd.DataFrame({'datetime': hourly_range})
    
    # For each day in original data, distribute values across 24 hours
    daily_data = []
    
    for _, row in df.iterrows():
        day_start = row['datetime'].normalize()
        
        # Create 24 hourly records for this day
        for hour in range(24):
            hour_dt = day_start + timedelta(hours=hour)
            
            hour_record = {
                'datetime': hour_dt,
                'outdoor_temp': row['outdoor_temp'],
                'outdoor_humidity': row['outdoor_humidity'],
                'outdoor_windspeed': row['outdoor_windspeed'],
                'solar': row['solar'],
            }
            
            # Add sunrise/sunset epochs if available
            if 'sunriseEpoch' in row.index:
                hour_record['sunriseEpoch'] = row['sunriseEpoch']
                hour_record['sunsetEpoch'] = row['sunsetEpoch']
            
            daily_data.append(hour_record)
    
    hourly_df = pd.DataFrame(daily_data)
    
    # Add diurnal variation to outdoor temp (simple sinusoidal pattern)
    # Peak at 14:00, minimum at 05:00
    hours = hourly_df['datetime'].dt.hour
    temp_variation = 3 * np.sin((hours - 5) * np.pi / 12)  # ±3°C variation
    hourly_df['outdoor_temp'] = hourly_df['outdoor_temp'] + temp_variation
    
    # Add diurnal variation to humidity (inverse of temp)
    humidity_variation = -5 * np.sin((hours - 5) * np.pi / 12)  # ±5% variation
    hourly_df['outdoor_humidity'] = hourly_df['outdoor_humidity'] + humidity_variation
    hourly_df['outdoor_humidity'] = hourly_df['outdoor_humidity'].clip(20, 100)
    
    # Distribute solar radiation across daylight hours
    # Solar should be 0 at night and peak around midday
    solar_pattern = np.maximum(0, np.sin((hours - 6) * np.pi / 12))
    hourly_df['solar'] = hourly_df['solar'] * solar_pattern
    
    # Calculate datetimeEpoch
    hourly_df['datetimeEpoch'] = hourly_df['datetime'].astype(np.int64) // 10**9
    
    return hourly_df

In [4]:
def add_day_night_flag(df):
    """
    Add day_night_flag column based on sunrise/sunset or hour heuristic.
    """
    if 'sunriseEpoch' in df.columns and 'sunsetEpoch' in df.columns:
        # Use actual sunrise/sunset times
        df['day_night_flag'] = (
            (df['datetimeEpoch'] >= df['sunriseEpoch']) & 
            (df['datetimeEpoch'] <= df['sunsetEpoch'])
        ).astype(int)
    else:
        # Use hour heuristic: day = 06:00-18:00
        hour = df['datetime'].dt.hour
        df['day_night_flag'] = ((hour >= 6) & (hour < 18)).astype(int)
    
    return df

In [5]:
def generate_passive_greenhouse_conditions(df):
    """
    Generate indoor greenhouse conditions based on passive greenhouse effects.
    No active control systems.
    """
    # A) Indoor Temperature
    deltaT = np.where(
        df['day_night_flag'] == 1,
        0.02 * df['solar'],  # Day: solar radiation effect
        1.5  # Night: modest heat retention
    )
    df['indoor_temp'] = df['outdoor_temp'] + deltaT
    
    # B) Indoor Humidity (clamped 30-100%)
    indoor_humidity = np.where(
        df['day_night_flag'] == 1,
        df['outdoor_humidity'] - (0.5 * deltaT) + 0.4,  # Day: drying effect
        df['outdoor_humidity'] + 5  # Night: moisture accumulation
    )
    df['indoor_humidity'] = np.clip(indoor_humidity, 30, 100)
    
    # C) Indoor Air Velocity
    df['indoor_air_velocity'] = df['outdoor_windspeed'] * 0.1
    
    # D) Indoor CO2 (clamped >= 300 ppm)
    indoor_co2 = np.where(
        df['day_night_flag'] == 1,
        400 - (0.05 * df['solar']),  # Day: photosynthesis depletes CO2
        440  # Night: respiration increases CO2
    )
    df['indoor_CO2'] = np.maximum(indoor_co2, 300)
    
    return df

In [6]:
def calculate_derived_features(df):
    """
    Calculate derived greenhouse features.
    """
    # E) Dew Point
    df['dew_point'] = df['indoor_temp'] - ((100 - df['indoor_humidity']) / 5)
    
    # F) VPD (Vapor Pressure Deficit)
    SVP = 0.6108 * np.exp((17.27 * df['indoor_temp']) / (df['indoor_temp'] + 237.3))
    AVP = SVP * (df['indoor_humidity'] / 100)
    df['vpd'] = SVP - AVP
    
    # G) Leaf Wetness Proxy
    # High humidity (>85%) for 3+ consecutive hours
    high_humidity = (df['indoor_humidity'] > 85).astype(int)
    df['leaf_wetness_proxy'] = (high_humidity.rolling(window=3, min_periods=1).sum() >= 3).astype(int)
    
    return df

In [7]:
def validate_and_save(df, output_path, year):
    """
    Validate dataset and save to CSV.
    """
    # Select final columns
    final_columns = [
        'datetime',
        'indoor_temp',
        'indoor_humidity',
        'indoor_air_velocity',
        'indoor_CO2',
        'solarradiation',
        'day_night_flag',
        'vpd',
        'dew_point',
        'leaf_wetness_proxy'
    ]
    
    # Rename solar back to solarradiation
    df = df.rename(columns={'solar': 'solarradiation'})
    
    df_final = df[final_columns].copy()
    
    # Validation
    print(f"\n{'='*60}")
    print(f"VALIDATION SUMMARY - {year}")
    print(f"{'='*60}")
    
    total_rows = len(df_final)
    unique_days = df_final['datetime'].dt.date.nunique()
    min_dt = df_final['datetime'].min()
    max_dt = df_final['datetime'].max()
    
    print(f"Total rows: {total_rows}")
    print(f"Unique days: {unique_days}")
    print(f"Date range: {min_dt} to {max_dt}")
    
    # Check 24 records per day
    records_per_day = df_final.groupby(df_final['datetime'].dt.date).size()
    if (records_per_day == 24).all():
        print(f"✓ All days have exactly 24 hourly records")
    else:
        problem_days = records_per_day[records_per_day != 24]
        print(f"✗ WARNING: {len(problem_days)} days do not have 24 records")
        print(f"  Problem days: {problem_days.to_dict()}")
    
    # Check for NaNs
    nan_counts = df_final.isnull().sum()
    if nan_counts.sum() == 0:
        print(f"✓ No missing values in any column")
    else:
        print(f"✗ WARNING: Missing values found:")
        print(nan_counts[nan_counts > 0])
    
    # Data ranges
    print(f"\nData Ranges:")
    print(f"  indoor_temp: {df_final['indoor_temp'].min():.1f} to {df_final['indoor_temp'].max():.1f} °C")
    print(f"  indoor_humidity: {df_final['indoor_humidity'].min():.1f} to {df_final['indoor_humidity'].max():.1f} %")
    print(f"  indoor_CO2: {df_final['indoor_CO2'].min():.1f} to {df_final['indoor_CO2'].max():.1f} ppm")
    print(f"  vpd: {df_final['vpd'].min():.2f} to {df_final['vpd'].max():.2f} kPa")
    print(f"  Day records: {(df_final['day_night_flag']==1).sum()} ({(df_final['day_night_flag']==1).sum()/len(df_final)*100:.1f}%)")
    print(f"  Leaf wetness events: {df_final['leaf_wetness_proxy'].sum()} hours ({df_final['leaf_wetness_proxy'].sum()/len(df_final)*100:.1f}%)")
    
    # Save
    df_final.to_csv(output_path, index=False)
    print(f"\n✓ Saved to: {output_path}")
    print(f"{'='*60}\n")
    
    return df_final

## Main Processing Pipeline

In [8]:
def process_year(input_path, output_path, year):
    """
    Complete processing pipeline for one year.
    """
    print(f"\n{'#'*60}")
    print(f"PROCESSING YEAR {year}")
    print(f"{'#'*60}")
    
    # Step 1: Load and standardize
    print("\n[1/6] Loading and standardizing data...")
    df = load_and_standardize(input_path)
    
    # Step 2: Expand to hourly
    print("\n[2/6] Expanding to hourly resolution...")
    df = expand_to_hourly(df)
    
    # Step 3: Add day/night flag
    print("\n[3/6] Adding day/night flag...")
    df = add_day_night_flag(df)
    day_hours = (df['day_night_flag'] == 1).sum()
    print(f"  Day hours: {day_hours}, Night hours: {len(df) - day_hours}")
    
    # Step 4: Generate passive greenhouse conditions
    print("\n[4/6] Generating passive greenhouse conditions...")
    df = generate_passive_greenhouse_conditions(df)
    
    # Step 5: Calculate derived features
    print("\n[5/6] Calculating derived features...")
    df = calculate_derived_features(df)
    
    # Step 6: Validate and save
    print("\n[6/6] Validating and saving...")
    df_final = validate_and_save(df, output_path, year)
    
    return df_final

## Process Both Years

In [ ]:
# Define paths
base_path = Path('../data/external/Weather Data')
output_base = Path('../data/processed/Greenhouse Indoor Conditions')
output_base.mkdir(parents=True, exist_ok=True)

# File paths for both years
years_config = [
    {
        'year': 2024,
        'input': base_path / 'dindigul_weather_2024.csv',
        'output': output_base / 'dindigul_greenhouse_indoor_2024.csv'
    },
    {
        'year': 2025,
        'input': base_path / 'dindigul_weather_2025.csv',
        'output': output_base / 'dindigul_greenhouse_indoor_2025.csv'
    }
]

# Process both years
results = {}
for config in years_config:
    results[config['year']] = process_year(
        config['input'],
        config['output'],
        config['year']
    )

print("\n" + "="*60)
print("ALL PROCESSING COMPLETE")
print("="*60)
print(f"\nGenerated files:")
for config in years_config:
    if config['output'].exists():
        size_mb = config['output'].stat().st_size / (1024*1024)
        print(f"  ✓ {config['output'].name} ({size_mb:.2f} MB)")
    else:
        print(f"  ✗ {config['output'].name} - NOT FOUND")


############################################################
PROCESSING YEAR 2024
############################################################

[1/6] Loading and standardizing data...
  Loaded 366 daily records from dindigul_weather_2024.csv

[2/6] Expanding to hourly resolution...
  Creating hourly index: 2024-01-01 00:00:00 to 2024-12-31 23:00:00
  Total hourly records: 8784

[3/6] Adding day/night flag...
  Day hours: 4435, Night hours: 4349

[4/6] Generating passive greenhouse conditions...

[5/6] Calculating derived features...

[6/6] Validating and saving...

VALIDATION SUMMARY - 2024
Total rows: 8784
Unique days: 366
Date range: 2024-01-01 00:00:00 to 2024-12-31 23:00:00
✓ All days have exactly 24 hourly records
✓ No missing values in any column

Data Ranges:
  indoor_temp: 19.8 to 42.8 °C
  indoor_humidity: 43.0 to 100.0 %
  indoor_CO2: 384.0 to 440.0 ppm
  vpd: 0.00 to 4.38 kPa
  Day records: 4435 (50.5%)
  Leaf wetness events: 864 hours (9.8%)

✓ Saved to: ..\data\processed\

## Quick Data Preview

In [10]:
# Display first few records of 2024 data
print("Sample from 2024 dataset (first 24 hours):")
results[2024].head(24)

Sample from 2024 dataset (first 24 hours):


,datetime,indoor_temp,indoor_humidity,indoor_air_velocity,indoor_CO2,solarradiation,day_night_flag,vpd,dew_point,leaf_wetness_proxy
0,2024-01-01 00:00:00,25.002223,83.729629,1.84,440.000000,0.000000e+00,0,0.515477,21.748148,0
1,2024-01-01 01:00:00,25.301924,83.230127,1.84,440.000000,0.000000e+00,0,0.540861,21.947949,0
2,2024-01-01 02:00:00,24.278680,77.835534,1.84,400.000000,0.000000e+00,1,0.672513,19.845786,0
3,2024-01-01 03:00:00,24.900000,76.800000,1.84,400.000000,0.000000e+00,1,0.730558,20.260000,0
4,2024-01-01 04:00:00,25.623543,75.594095,1.84,400.000000,0.000000e+00,1,0.802309,20.742362,0
5,2024-01-01 05:00:00,26.400000,74.300000,1.84,400.000000,0.000000e+00,1,0.884529,21.260000,0
6,2024-01-01 06:00:00,27.176457,73.005905,1.84,400.000000,0.000000e+00,1,0.972440,21.777638,0
7,2024-01-01 07:00:00,28.888171,71.305914,1.84,397.529572,4.940856e+01,1,1.141985,23.149354,0
8,2024-01-01 08:00:00,30.430320,69.809966,1.84,395.227500,9.545000e+01,1,1.312942,24.392314,0
9,2024-01-01 09:00:00,31.697810,68.620006,1.84,393.250666,1.349867e+02,1,1.466758,25.421811,0


In [11]:
# Display statistics
print("Statistical Summary - 2024:")
results[2024][['indoor_temp', 'indoor_humidity', 'indoor_CO2', 'vpd', 'dew_point']].describe()

Statistical Summary - 2024:


,indoor_temp,indoor_humidity,indoor_CO2,vpd,dew_point
count,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000
mean,30.537087,71.850553,417.756640,1.341484,24.907197
std,3.814208,10.932567,22.279669,0.777833,2.570237
min,19.801924,43.043194,383.990000,0.000000,17.845786
25%,27.700000,63.723235,397.719304,0.755622,23.047409
50%,30.102223,71.600000,400.000000,1.215001,24.728049
75%,32.998557,80.064466,440.000000,1.744411,26.695518
max,42.751777,100.000000,440.000000,4.377717,33.120452


In [12]:
# Display sample from a specific day to verify hourly structure
sample_day = results[2024]['datetime'].dt.date.unique()[10]  # 11th day
print(f"\nVerification: All 24 hours of {sample_day}")
day_data = results[2024][results[2024]['datetime'].dt.date == sample_day]
print(f"Records for this day: {len(day_data)}")
day_data[['datetime', 'indoor_temp', 'indoor_humidity', 'day_night_flag', 'solarradiation', 'vpd']]


Verification: All 24 hours of 2024-01-11
Records for this day: 24


,datetime,indoor_temp,indoor_humidity,day_night_flag,solarradiation,vpd
240,2024-01-11 00:00:00,25.402223,80.329629,0,0.000000e+00,0.638199
241,2024-01-11 01:00:00,25.701924,79.830127,0,0.000000e+00,0.666143
242,2024-01-11 02:00:00,24.678680,74.435534,1,0.000000e+00,0.794455
243,2024-01-11 03:00:00,25.300000,73.400000,1,0.000000e+00,0.857804
244,2024-01-11 04:00:00,26.023543,72.194095,1,0.000000e+00,0.935980
245,2024-01-11 05:00:00,26.800000,70.900000,1,0.000000e+00,1.025402
246,2024-01-11 06:00:00,27.576457,69.605905,1,0.000000e+00,1.120845
247,2024-01-11 07:00:00,29.280924,67.909538,1,4.904621e+01,1.306457
248,2024-01-11 08:00:00,30.816320,66.416966,1,9.475000e+01,1.493043
249,2024-01-11 09:00:00,32.078011,65.229906,1,1.339967e+02,1.660542
